### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

### Create Output Folder

In [2]:
output_dir = Path("data/synthetic")

output_dir.mkdir(parents=True, exist_ok=True)

print("Output folder created successfully!")

Output folder created successfully!


### Generate organizations.csv

In [3]:
np.random.seed(42)

num_organizations = 1500

industries = [
    "Technology",
    "Finance",
    "Healthcare",
    "E-commerce",
    "Education",
    "Retail"
]

company_sizes = [
    "Startup",
    "Small",
    "Medium",
    "Large",
    "Enterprise"
]

countries = [
    "USA",
    "UK",
    "Canada",
    "Germany",
    "India",
    "Bangladesh",
    "Singapore",
    "Japan"
]

organizations = pd.DataFrame({
    "organization_id": range(1, num_organizations + 1),
    "organization_name": [
        f"Organization_{i}" for i in range(1, num_organizations + 1)
    ],
    "industry": np.random.choice(industries, num_organizations),
    "company_size": np.random.choice(company_sizes, num_organizations),
    "created_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2024-01-01", "2025-12-31"),
            num_organizations
        )
    ),
    "country": np.random.choice(countries, num_organizations)
})

organizations.head()

,organization_id,organization_name,industry,company_size,created_date,country
0,1,Organization_1,E-commerce,Small,2024-05-22,USA
1,2,Organization_2,Education,Large,2024-07-09,Bangladesh
2,3,Organization_3,Healthcare,Medium,2025-06-16,Canada
3,4,Organization_4,Education,Enterprise,2024-08-14,Japan
4,5,Organization_5,Education,Large,2024-11-08,UK


### CSV save

In [4]:
organizations.to_csv(
    output_dir / "organizations.csv",
    index=False
)

print("organizations.csv created successfully!")

organizations.csv created successfully!


### Generate users.csv

In [5]:
num_users = 10000

user_roles = [
    "Developer",
    "Admin",
    "Product Manager",
    "Data Analyst",
    "Founder"
]

acquisition_channels = [
    "Organic",
    "Paid Ads",
    "Referral",
    "Social Media",
    "Direct"
]

organization_dates = organizations.set_index(
    "organization_id"
)["created_date"]

user_organization_ids = np.random.choice(
    organizations["organization_id"],
    num_users
)

signup_dates = []

for org_id in user_organization_ids:
    start_date = organization_dates.loc[org_id]

    signup_date = start_date + pd.to_timedelta(
        np.random.randint(0, 181),
        unit="D"
    )

    signup_dates.append(signup_date)

users = pd.DataFrame({
    "user_id": range(1, num_users + 1),
    "organization_id": user_organization_ids,
    "signup_date": pd.to_datetime(signup_dates),
    "country": np.random.choice(countries, num_users),
    "user_role": np.random.choice(user_roles, num_users),
    "acquisition_channel": np.random.choice(
        acquisition_channels,
        num_users
    ),
    "is_verified": np.random.choice(
        [0, 1],
        num_users,
        p=[0.2, 0.8]
    )
})

users.head()

,user_id,organization_id,signup_date,country,user_role,acquisition_channel,is_verified
0,1,283,2025-04-03,USA,Founder,Social Media,1
1,2,591,2024-06-23,Germany,Admin,Referral,1
2,3,72,2025-12-27,Japan,Founder,Referral,1
3,4,1287,2024-12-28,Canada,Admin,Paid Ads,0
4,5,703,2025-10-14,Germany,Founder,Paid Ads,1


### CSV save

In [6]:
users.to_csv(
    output_dir / "users.csv",
    index=False
)

print("users.csv created successfully!")

users.csv created successfully!


### Validate CSV Files

In [7]:
print("Organizations Shape:", organizations.shape)
print("Users Shape:", users.shape)

print("\nOrganizations Missing Values:")
print(organizations.isnull().sum())

print("\nUsers Missing Values:")
print(users.isnull().sum())

print("\nDuplicate Organization IDs:",
      organizations["organization_id"].duplicated().sum())

print("Duplicate User IDs:",
      users["user_id"].duplicated().sum())

print("Invalid Organization References:",
      (~users["organization_id"].isin(
          organizations["organization_id"]
      )).sum())

Organizations Shape: (1500, 6)
Users Shape: (10000, 7)

Organizations Missing Values:
organization_id      0
organization_name    0
industry             0
company_size         0
created_date         0
country              0
dtype: int64

Users Missing Values:
user_id                0
organization_id        0
signup_date            0
country                0
user_role              0
acquisition_channel    0
is_verified            0
dtype: int64

Duplicate Organization IDs: 0
Duplicate User IDs: 0
Invalid Organization References: 0


### Check Generated CSV Files

In [8]:
print("Generated CSV Files:")

for file in output_dir.glob("*.csv"):
    print(file.name)

Generated CSV Files:
organizations.csv
users.csv


### Generate subscriptions.csv

In [9]:
np.random.seed(42)

num_subscriptions = len(organizations)

plan_types = ["Free", "Pro", "Enterprise"]
subscription_statuses = ["Active", "Cancelled", "Expired"]

plan_prices = {
    "Free": 0.00,
    "Pro": 49.00,
    "Enterprise": 199.00
}

subscriptions = pd.DataFrame({
    "subscription_id": range(1, num_subscriptions + 1),
    "organization_id": organizations["organization_id"],
    "plan_type": np.random.choice(
        plan_types,
        num_subscriptions,
        p=[0.40, 0.40, 0.20]
    ),
    "subscription_status": np.random.choice(
        subscription_statuses,
        num_subscriptions,
        p=[0.70, 0.15, 0.15]
    )
})

subscriptions["monthly_price"] = (
    subscriptions["plan_type"].map(plan_prices)
)

organization_dates = organizations.set_index(
    "organization_id"
)["created_date"]

subscriptions["start_date"] = [
    org_date + pd.to_timedelta(
        np.random.randint(0, 181), unit="D"
    )
    for org_date in organizations["created_date"]
]

subscriptions["end_date"] = pd.NaT

for i in subscriptions.index:
    if subscriptions.loc[i, "subscription_status"] != "Active":
        start = subscriptions.loc[i, "start_date"]
        max_days = max(1, (pd.Timestamp("2026-11-15") - start).days)

        duration = np.random.randint(
            1, min(max_days, 181) + 1
        )

        subscriptions.loc[i, "end_date"] = (
            start + pd.Timedelta(days=duration)
        )

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"]
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"]
)

subscriptions.head()

,subscription_id,organization_id,plan_type,subscription_status,monthly_price,start_date,end_date
0,1,1,Free,Active,0.0,2024-09-13,NaT
1,2,2,Enterprise,Active,199.0,2024-09-03,NaT
2,3,3,Pro,Active,49.0,2025-08-26,NaT
3,4,4,Pro,Active,49.0,2024-08-20,NaT
4,5,5,Free,Active,0.0,2025-03-06,NaT


### Save CSV

In [10]:
subscriptions.to_csv(
    output_dir / "subscriptions.csv",
    index=False
)

print("subscriptions.csv created successfully!")

subscriptions.csv created successfully!


### Generate product_events.csv

In [11]:
np.random.seed(42)

num_events = 80000

event_names = [
    "signup",
    "login",
    "api_key_created",
    "first_api_request",
    "sdk_integrated",
    "monitoring_enabled",
    "documentation_viewed",
    "webhook_created",
    "analytics_dashboard_viewed",
    "subscription_upgraded"
]

event_user_ids = np.random.choice(
    users["user_id"],
    num_events
)

user_signup_dates = users.set_index(
    "user_id"
)["signup_date"]

event_dates = []

for user_id in event_user_ids:
    signup_date = user_signup_dates.loc[user_id]

    random_days = np.random.randint(0, 181)

    event_date = signup_date + pd.Timedelta(
        days=random_days
    )

    event_dates.append(event_date)

product_events = pd.DataFrame({
    "event_id": range(1, num_events + 1),
    "user_id": event_user_ids,
    "event_name": np.random.choice(
        event_names,
        num_events
    ),
    "event_timestamp": pd.to_datetime(event_dates),
    "session_id": [
        f"SESSION_{i}" for i in range(1, num_events + 1)
    ],
    "platform": np.random.choice(
        ["Web", "IDE", "API"],
        num_events,
        p=[0.50, 0.30, 0.20]
    ),
    "device_type": np.random.choice(
        ["Desktop", "Mobile"],
        num_events,
        p=[0.80, 0.20]
    ),
    "event_properties": np.random.choice(
        [
            "success",
            "failure",
            "page_view",
            "feature_interaction",
            "integration_action"
        ],
        num_events
    )
})

product_events.head()

,event_id,user_id,event_name,event_timestamp,session_id,platform,device_type,event_properties
0,1,7271,webhook_created,2025-12-01,SESSION_1,Web,Desktop,integration_action
1,2,861,documentation_viewed,2024-08-03,SESSION_2,API,Desktop,page_view
2,3,5391,sdk_integrated,2025-02-08,SESSION_3,Web,Desktop,integration_action
3,4,5192,documentation_viewed,2024-03-26,SESSION_4,API,Desktop,integration_action
4,5,5735,signup,2026-04-05,SESSION_5,Web,Mobile,page_view


### Save CSV

In [12]:
product_events.to_csv(
    output_dir / "product_events.csv",
    index=False
)

print("product_events.csv created successfully!")

product_events.csv created successfully!


### Generate feature usage

In [13]:
np.random.seed(42)

features = [
    "API Monitoring",
    "SDK Integration",
    "Analytics Dashboard",
    "Webhooks",
    "Rate Limiting",
    "API Authentication",
    "API Documentation"
]

feature_records = []

for user_id in users["user_id"]:

    signup_date = user_signup_dates.loc[user_id]

    num_features = np.random.randint(1, 4)

    selected_features = np.random.choice(
        features,
        num_features,
        replace=False
    )

    for feature in selected_features:

        first_used = signup_date + pd.Timedelta(
            days=np.random.randint(0, 151)
        )

        usage_count = np.random.randint(1, 501)

        last_used = first_used + pd.Timedelta(
            days=np.random.randint(0, 61)
        )

        feature_records.append({
            "user_id": user_id,
            "feature_name": feature,
            "first_used_at": first_used,
            "usage_count": usage_count,
            "last_used_at": last_used
        })

feature_usage = pd.DataFrame(feature_records)

feature_usage.insert(
    0,
    "usage_id",
    range(1, len(feature_usage) + 1)
)

feature_usage.head()

,usage_id,user_id,feature_name,first_used_at,usage_count,last_used_at
0,1,1,SDK Integration,2025-07-14,122,2025-08-01
1,2,1,API Authentication,2025-06-16,459,2025-07-09
2,3,1,API Monitoring,2025-07-28,100,2025-09-05
3,4,2,API Monitoring,2024-10-30,192,2024-12-28
4,5,2,Analytics Dashboard,2024-07-13,161,2024-07-24


### Save CSV

In [14]:
feature_usage.to_csv(
    output_dir / "feature_usage.csv",
    index=False
)

print("feature_usage.csv created successfully!")

feature_usage.csv created successfully!


### Generate api_usage.csv

In [15]:
np.random.seed(42)

num_api_records = 50000

api_user_ids = np.random.choice(
    users["user_id"],
    num_api_records
)

api_dates = []

for user_id in api_user_ids:

    signup_date = user_signup_dates.loc[user_id]

    request_date = signup_date + pd.Timedelta(
        days=np.random.randint(0, 181)
    )

    api_dates.append(request_date)

request_counts = np.random.randint(
    10,
    1001,
    num_api_records
)

error_counts = np.array([
    np.random.binomial(n=request, p=0.05)
    for request in request_counts
])

success_counts = request_counts - error_counts

endpoints = [
    "/api/v1/users",
    "/api/v1/products",
    "/api/v1/orders",
    "/api/v1/analytics",
    "/api/v1/auth",
    "/api/v1/webhooks"
]

api_usage = pd.DataFrame({
    "api_usage_id": range(1, num_api_records + 1),
    "user_id": api_user_ids,
    "request_timestamp": pd.to_datetime(api_dates),
    "endpoint": np.random.choice(
        endpoints,
        num_api_records
    ),
    "request_count": request_counts,
    "success_count": success_counts,
    "error_count": error_counts,
    "response_time_ms": np.round(
        np.random.uniform(20, 1000, num_api_records),
        2
    )
})

api_usage.head()

,api_usage_id,user_id,request_timestamp,endpoint,request_count,success_count,error_count,response_time_ms
0,1,7271,2025-12-01,/api/v1/products,318,302,16,622.86
1,2,861,2024-09-19,/api/v1/analytics,1000,941,59,29.98
2,3,5391,2025-02-20,/api/v1/users,40,40,0,911.65
3,4,5192,2024-06-28,/api/v1/products,861,813,48,881.71
4,5,5735,2025-12-10,/api/v1/orders,236,225,11,868.30


### Save CSV

In [16]:
api_usage.to_csv(
    output_dir / "api_usage.csv",
    index=False
)

print("api_usage.csv created successfully!")

api_usage.csv created successfully!


### Correct Date Ranges

In [17]:
MAX_DATE = pd.Timestamp("2026-11-15")

product_events["event_timestamp"] = (
    product_events["event_timestamp"].clip(upper=MAX_DATE)
)

api_usage["request_timestamp"] = (
    api_usage["request_timestamp"].clip(upper=MAX_DATE)
)

feature_usage["first_used_at"] = (
    feature_usage["first_used_at"].clip(upper=MAX_DATE)
)

feature_usage["last_used_at"] = (
    feature_usage["last_used_at"].clip(upper=MAX_DATE)
)

# Last usage cannot be earlier than first usage
feature_usage["last_used_at"] = feature_usage[
    ["first_used_at", "last_used_at"]
].max(axis=1)

product_events.to_csv(
    output_dir / "product_events.csv",
    index=False
)

feature_usage.to_csv(
    output_dir / "feature_usage.csv",
    index=False
)

api_usage.to_csv(
    output_dir / "api_usage.csv",
    index=False
)

print("Date ranges corrected successfully!")

Date ranges corrected successfully!


### Generate experiment_assignments.csv

In [18]:
np.random.seed(42)

eligible_users = users[
    users["signup_date"] <= pd.Timestamp("2026-10-01")
].copy()

experiment_users = eligible_users.sample(
    n=6000,
    random_state=42
).copy()

experiment_users = experiment_users.reset_index(
    drop=True
)

assigned_dates = []

cutoff_date = pd.Timestamp("2026-10-31")

for signup_date in experiment_users["signup_date"]:

    available_days = max(
        0,
        (cutoff_date - signup_date).days
    )

    random_days = np.random.randint(
        0,
        available_days + 1
    )

    assigned_dates.append(
        signup_date + pd.Timedelta(days=random_days)
    )

experiment_assignments = pd.DataFrame({
    "assignment_id": range(1, 6001),
    "experiment_id": "EXP_FEATURE_ONBOARDING_001",
    "user_id": experiment_users["user_id"],
    "variant": np.random.choice(
        ["Control", "Treatment"],
        6000,
        p=[0.50, 0.50]
    ),
    "assigned_at": pd.to_datetime(assigned_dates)
})

experiment_assignments.head()

,assignment_id,experiment_id,user_id,variant,assigned_at
0,1,EXP_FEATURE_ONBOARDING_001,6253,Control,2025-08-05
1,2,EXP_FEATURE_ONBOARDING_001,4685,Treatment,2026-10-30
2,3,EXP_FEATURE_ONBOARDING_001,1732,Control,2026-10-25
3,4,EXP_FEATURE_ONBOARDING_001,4743,Treatment,2025-10-19
4,5,EXP_FEATURE_ONBOARDING_001,4522,Control,2024-10-07


### Save CSV

In [19]:
experiment_assignments.to_csv(
    output_dir / "experiment_assignments.csv",
    index=False
)

print("experiment_assignments.csv created successfully!")

experiment_assignments.csv created successfully!


### Generate experiment_outcomes.csv

In [20]:
np.random.seed(42)

num_outcomes = len(experiment_assignments)

outcomes = experiment_assignments[
    ["assignment_id", "user_id", "variant", "assigned_at"]
].copy()

is_treatment = (
    outcomes["variant"] == "Treatment"
).to_numpy()

# Simulated outcome probabilities
sdk_prob = np.where(is_treatment, 0.75, 0.60)

api_prob = np.where(is_treatment, 0.70, 0.55)

adoption_prob = np.where(is_treatment, 0.65, 0.45)

retention_prob = np.where(is_treatment, 0.60, 0.50)

experiment_outcomes = pd.DataFrame({
    "outcome_id": range(1, num_outcomes + 1),
    "assignment_id": outcomes["assignment_id"],
    "user_id": outcomes["user_id"],
    "sdk_integrated": np.random.binomial(
        1, sdk_prob
    ),
    "first_api_request": np.random.binomial(
        1, api_prob
    ),
    "adopted_feature": np.random.binomial(
        1, adoption_prob
    ),
    "retained_d7": np.random.binomial(
        1, retention_prob
    ),
    "measured_at": (
        outcomes["assigned_at"] +
        pd.Timedelta(days=7)
    )
})

experiment_outcomes.head()

,outcome_id,assignment_id,user_id,sdk_integrated,first_api_request,adopted_feature,retained_d7,measured_at
0,1,1,6253,1,0,1,1,2025-08-12
1,2,2,4685,0,1,1,1,2026-11-06
2,3,3,1732,0,0,0,0,2026-11-01
3,4,4,4743,1,1,0,0,2025-10-26
4,5,5,4522,1,1,0,1,2024-10-14


### Save CSV

In [21]:
experiment_outcomes.to_csv(
    output_dir / "experiment_outcomes.csv",
    index=False
)

print("experiment_outcomes.csv created successfully!")

experiment_outcomes.csv created successfully!


### Validate All 8 CSV Files

In [22]:
csv_files = [
    "organizations.csv",
    "users.csv",
    "subscriptions.csv",
    "product_events.csv",
    "feature_usage.csv",
    "api_usage.csv",
    "experiment_assignments.csv",
    "experiment_outcomes.csv"
]

for file in csv_files:

    df = pd.read_csv(output_dir / file)

    print(f"\nFile: {file}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing Values:", df.isnull().sum().sum())


File: organizations.csv
Rows: 1500
Columns: 6
Missing Values: 0

File: users.csv
Rows: 10000
Columns: 7
Missing Values: 0

File: subscriptions.csv
Rows: 1500
Columns: 7
Missing Values: 1043

File: product_events.csv
Rows: 80000
Columns: 8
Missing Values: 0

File: feature_usage.csv
Rows: 20041
Columns: 6
Missing Values: 0

File: api_usage.csv
Rows: 50000
Columns: 8
Missing Values: 0

File: experiment_assignments.csv
Rows: 6000
Columns: 5
Missing Values: 0

File: experiment_outcomes.csv
Rows: 6000
Columns: 8
Missing Values: 0


### Check duplicate primary keys

In [23]:
primary_keys = {
    "organizations.csv": "organization_id",
    "users.csv": "user_id",
    "subscriptions.csv": "subscription_id",
    "product_events.csv": "event_id",
    "feature_usage.csv": "usage_id",
    "api_usage.csv": "api_usage_id",
    "experiment_assignments.csv": "assignment_id",
    "experiment_outcomes.csv": "outcome_id"
}

for file, key in primary_keys.items():

    df = pd.read_csv(output_dir / file)

    print(
        file,
        "Duplicate IDs:",
        df[key].duplicated().sum()
    )

organizations.csv Duplicate IDs: 0
users.csv Duplicate IDs: 0
subscriptions.csv Duplicate IDs: 0
product_events.csv Duplicate IDs: 0
feature_usage.csv Duplicate IDs: 0
api_usage.csv Duplicate IDs: 0
experiment_assignments.csv Duplicate IDs: 0
experiment_outcomes.csv Duplicate IDs: 0


### Check relationships

In [24]:
print(
    "Invalid User References in Events:",
    (~product_events["user_id"].isin(users["user_id"])).sum()
)

print(
    "Invalid User References in Feature Usage:",
    (~feature_usage["user_id"].isin(users["user_id"])).sum()
)

print(
    "Invalid User References in API Usage:",
    (~api_usage["user_id"].isin(users["user_id"])).sum()
)

print(
    "Invalid Organization References in Subscriptions:",
    (~subscriptions["organization_id"].isin(
        organizations["organization_id"]
    )).sum()
)

print(
    "Invalid Assignment References in Outcomes:",
    (~experiment_outcomes["assignment_id"].isin(
        experiment_assignments["assignment_id"]
    )).sum()
)

Invalid User References in Events: 0
Invalid User References in Feature Usage: 0
Invalid User References in API Usage: 0
Invalid Organization References in Subscriptions: 0
Invalid Assignment References in Outcomes: 0
